In [0]:
%pip install flask flask-cors pyngrok

  Attempting uninstall: blinker
    Found existing installation: blinker 1.7.0
    Not uninstalling blinker at /usr/lib/python3/dist-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-7ff872ad-2579-4988-8336-21c6b7dd8862
    Can't uninstall 'blinker'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
backend_code = '''
from flask import Flask, request, jsonify
from flask_cors import CORS
import json, math, requests, os, re

app = Flask(__name__)
CORS(app)

FACILITIES = []

DATABRICKS_TOKEN = os.environ.get("DATABRICKS_TOKEN","")
WORKSPACE_URL = os.environ.get("WORKSPACE_URL","")

def call_llm(prompt):
    r = requests.post(
        f"https://{WORKSPACE_URL}/serving-endpoints/databricks-llama-4-maverick/invocations",
        headers={"Authorization": f"Bearer {DATABRICKS_TOKEN}"},
        json={"messages":[{"role":"user","content":prompt}],"max_tokens":600}
    )
    return r.json()["choices"][0]["message"]["content"]

def haversine(lat1,lon1,lat2,lon2):
    R=6371
    p1,p2=math.radians(lat1),math.radians(lat2)
    dp=math.radians(lat2-lat1)
    dl=math.radians(lon2-lon1)
    a=math.sin(dp/2)**2+math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return R*2*math.atan2(math.sqrt(a),math.sqrt(1-a))

@app.route("/triage", methods=["POST"])
def triage():
    body = request.json
    symptoms = body.get("symptoms","")
    location = body.get("location","India")
    prompt = f"""You are AarogyaPath emergency triage for rural India.
Patient location: {location}
Symptoms: {symptoms}
Return ONLY valid JSON no markdown:
{{
  "severity": "EMERGENCY or URGENT or MILD",
  "severity_reason": "one sentence why",
  "action": "what to do RIGHT NOW",
  "facility_type_needed": "ICU or Emergency or GeneralSurgery or Dialysis or Clinic or Homecare",
  "call_ambulance": true or false,
  "home_care_steps": ["step1","step2","step3"],
  "red_flags": ["flag1","flag2"],
  "chain_of_thought": "your reasoning"
}}"""
    raw = call_llm(prompt).strip()
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if match: raw = match.group()
    return jsonify(json.loads(raw))

@app.route("/facilities", methods=["POST"])
def find_facilities():
    body = request.json
    patient_lat = float(body.get("lat", 20.5))
    patient_lon = float(body.get("lon", 78.9))
    needed_type = body.get("facility_type","Emergency")

    scored = []
    for f in FACILITIES:
        try:
            lat = float(f.get("latitude") or 0)
            lon = float(f.get("longitude") or 0)
            if not lat or not lon: continue
            dist = haversine(patient_lat, patient_lon, lat, lon)
            trust = int(f.get("trust_score") or 50)
            reachability = max(0, 100 - dist * 0.5)
            cap_map = {
                "ICU":["has_icu","has_emergency"],
                "Emergency":["has_emergency"],
                "GeneralSurgery":["has_surgery","has_anesthesiologist"],
                "Dialysis":["has_dialysis"],
            }
            required = cap_map.get(needed_type,[])
            cap_score = (sum(1 for k in required if str(f.get(k,"")).lower()=="true")/len(required)*100) if required else 70
            last_mile = (trust*0.45)+(reachability*0.30)+(cap_score*0.25)
            scored.append({
                "facility_name": f.get("facility_name",""),
                "city": f.get("address_city",""),
                "state": f.get("state",""),
                "phone": f.get("phone",""),
                "distance_km": round(dist,1),
                "trust_score": trust,
                "trust_level": f.get("trust_level",""),
                "trust_flags": f.get("trust_flags",[]),
                "last_mile_score": round(last_mile,1),
                "has_emergency": f.get("has_emergency"),
                "has_icu": f.get("has_icu"),
                "has_surgery": f.get("has_surgery"),
                "citation": f.get("citation",""),
                "verdict": "RECOMMENDED" if last_mile>=70 else "USE IF NEEDED" if last_mile>=45 else "AVOID",
                "latitude": lat,
                "longitude": lon
            })
        except: continue
    scored.sort(key=lambda x: x["last_mile_score"], reverse=True)
    return jsonify({"facilities": scored[:5]})

@app.route("/deserts", methods=["GET"])
def deserts():
    from collections import defaultdict
    pins = defaultdict(lambda: {"missing":[],"state":"","lat":0,"lon":0})
    for f in FACILITIES:
        if int(f.get("trust_score") or 0) < 45: continue
        pin = str(f.get("pincode",""))
        missing = []
        if str(f.get("has_icu","")).lower()!="true": missing.append("ICU")
        if str(f.get("has_emergency","")).lower()!="true": missing.append("Emergency")
        if str(f.get("has_surgery","")).lower()!="true": missing.append("Surgery")
        if str(f.get("has_dialysis","")).lower()!="true": missing.append("Dialysis")
        if len(missing) >= 3:
            pins[pin]["missing"] = missing
            pins[pin]["state"] = f.get("state","")
            pins[pin]["lat"] = float(f.get("latitude") or 0)
            pins[pin]["lon"] = float(f.get("longitude") or 0)
    deserts_list = [
        {"pincode":k,"state":v["state"],
         "missing_capabilities":v["missing"],
         "severity":"CRITICAL" if len(v["missing"])>=4 else "HIGH",
         "lat":v["lat"],"lon":v["lon"]}
        for k,v in pins.items()
    ]
    deserts_list.sort(key=lambda x: len(x["missing_capabilities"]), reverse=True)
    return jsonify({
        "total_desert_pins": len(deserts_list),
        "critical_count": sum(1 for d in deserts_list if d["severity"]=="CRITICAL"),
        "deserts": deserts_list[:100]
    })

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status":"ok","facilities":len(FACILITIES)})
'''

with open("/tmp/app.py", "w") as f:
    f.write(backend_code)
print("✅ app.py written!")

✅ app.py written!


<>:1: SyntaxWarning: invalid escape sequence '\{'
<>:1: SyntaxWarning: invalid escape sequence '\{'
/home/spark-7ff872ad-2579-4988-8336-21/.ipykernel/3221/command-7249238166546848-1433876544:1: SyntaxWarning: invalid escape sequence '\{'
  backend_code = '''
<unknown>:1: SyntaxWarning: invalid escape sequence '\{'


In [0]:
%pip install openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
%pip install openpyxl requests pandas

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import requests
import pandas as pd
import json
from io import BytesIO

# Reload dataset
SHEET_ID = "1ZDuDmoQlyxZIEahDBlrMjf2wiWG7xU81"
url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=xlsx"
response = requests.get(url)
df = pd.read_excel(BytesIO(response.content))
print(f"✅ {len(df)} rows loaded!")

# Rule-based extraction — instant, no LLM needed
results = []
for _, row in df.iterrows():
    capability = str(row.get("capability", "") or "").lower()
    specialties = str(row.get("specialties", "") or "").lower()
    description = str(row.get("description", "") or "").lower()
    procedures = str(row.get("procedure", "") or "").lower()
    all_text = capability + " " + specialties + " " + description + " " + procedures

    has_emergency = any(w in all_text for w in ["emergency","24x7","24/7","always open","round the clock"])
    has_icu = any(w in all_text for w in ["icu","intensive care","critical care","ventilator"])
    has_surgery = any(w in all_text for w in ["surgery","surgical","operation","theatre","appendectomy"])
    has_dialysis = any(w in all_text for w in ["dialysis","renal","kidney"])
    has_neonatal = any(w in all_text for w in ["neonatal","nicu","newborn","neo natal"])
    has_oncology = any(w in all_text for w in ["oncology","cancer","chemotherapy","radiation"])
    has_anesthesiologist = any(w in all_text for w in ["anesthes","anaesth"])

    score = 100
    flags = []
    if has_surgery and not has_anesthesiologist:
        score -= 40
        flags.append("CRITICAL: Surgery claimed but no Anesthesiologist")
    if has_icu and "bed" not in all_text:
        score -= 20
        flags.append("WARNING: ICU claimed but no beds mentioned")
    if not has_emergency and not has_icu and not has_surgery:
        score -= 15
        flags.append("INFO: No high-acuity capabilities found")

    trust_level = "HIGH" if score >= 75 else "MEDIUM" if score >= 45 else "LOW"

    results.append({
        "facility_name": str(row.get("name", "")),
        "address_city": str(row.get("address_city", "")),
        "state": str(row.get("address_stateOrRegion", "")),
        "pincode": str(row.get("address_zipOrPostcode", "")),
        "latitude": float(row.get("latitude", 0) or 0),
        "longitude": float(row.get("longitude", 0) or 0),
        "phone": str(row.get("officialPhone", "") or ""),
        "facility_type": str(row.get("facilityTypeId", "") or ""),
        "has_emergency": has_emergency,
        "has_icu": has_icu,
        "has_surgery": has_surgery,
        "has_dialysis": has_dialysis,
        "has_neonatal": has_neonatal,
        "has_oncology": has_oncology,
        "has_anesthesiologist": has_anesthesiologist,
        "trust_score": score,
        "trust_level": trust_level,
        "trust_flags": flags,
        "citation": str(row.get("capability", "") or "")[:200],
        "extraction_confidence": 0.8
    })

print(f"✅ ALL {len(results)} facilities extracted instantly!")
print(f"🚨 Emergency: {sum(1 for r in results if r.get('has_emergency'))}")
print(f"🛏️  ICU:       {sum(1 for r in results if r.get('has_icu'))}")
print(f"🔪 Surgery:   {sum(1 for r in results if r.get('has_surgery'))}")
print(f"💉 Dialysis:  {sum(1 for r in results if r.get('has_dialysis'))}")
print(f"✅ HIGH trust: {sum(1 for r in results if r.get('trust_level')=='HIGH')}")
print(f"⚠️  MEDIUM:    {sum(1 for r in results if r.get('trust_level')=='MEDIUM')}")
print(f"❌ LOW:        {sum(1 for r in results if r.get('trust_level')=='LOW')}")

✅ 10000 rows loaded!
✅ ALL 10000 facilities extracted instantly!
🚨 Emergency: 1208
🛏️  ICU:       330
🔪 Surgery:   2342
💉 Dialysis:  227
✅ HIGH trust: 7751
⚠️  MEDIUM:    2128
❌ LOW:        121


In [0]:
import pandas as pd

results_df = pd.DataFrame(results)
spark_df = spark.createDataFrame(results_df.astype(str))
spark_df.write.mode("overwrite").saveAsTable("extracted_facilities")
print(f"✅ Saved to Spark table!")

✅ Saved to Spark table!


In [0]:
%pip install folium

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import requests
import pandas as pd
import json
from io import BytesIO

# Reload dataset
SHEET_ID = "1ZDuDmoQlyxZIEahDBlrMjf2wiWG7xU81"
url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=xlsx"
response = requests.get(url)
df = pd.read_excel(BytesIO(response.content))

# Rebuild results instantly
results = []
for _, row in df.iterrows():
    capability = str(row.get("capability", "") or "").lower()
    specialties = str(row.get("specialties", "") or "").lower()
    description = str(row.get("description", "") or "").lower()
    procedures = str(row.get("procedure", "") or "").lower()
    all_text = capability + " " + specialties + " " + description + " " + procedures

    has_emergency = any(w in all_text for w in ["emergency","24x7","24/7","always open","round the clock"])
    has_icu = any(w in all_text for w in ["icu","intensive care","critical care","ventilator"])
    has_surgery = any(w in all_text for w in ["surgery","surgical","operation","theatre","appendectomy"])
    has_dialysis = any(w in all_text for w in ["dialysis","renal","kidney"])
    has_neonatal = any(w in all_text for w in ["neonatal","nicu","newborn","neo natal"])
    has_oncology = any(w in all_text for w in ["oncology","cancer","chemotherapy","radiation"])
    has_anesthesiologist = any(w in all_text for w in ["anesthes","anaesth"])

    score = 100
    flags = []
    if has_surgery and not has_anesthesiologist:
        score -= 40
        flags.append("CRITICAL: Surgery claimed but no Anesthesiologist")
    if has_icu and "bed" not in all_text:
        score -= 20
        flags.append("WARNING: ICU claimed but no beds mentioned")
    if not has_emergency and not has_icu and not has_surgery:
        score -= 15
        flags.append("INFO: No high-acuity capabilities found")

    trust_level = "HIGH" if score >= 75 else "MEDIUM" if score >= 45 else "LOW"

    results.append({
        "facility_name": str(row.get("name", "")),
        "address_city": str(row.get("address_city", "")),
        "state": str(row.get("address_stateOrRegion", "")),
        "pincode": str(row.get("address_zipOrPostcode", "")),
        "latitude": float(row.get("latitude", 0) or 0),
        "longitude": float(row.get("longitude", 0) or 0),
        "phone": str(row.get("officialPhone", "") or ""),
        "facility_type": str(row.get("facilityTypeId", "") or ""),
        "has_emergency": has_emergency,
        "has_icu": has_icu,
        "has_surgery": has_surgery,
        "has_dialysis": has_dialysis,
        "has_neonatal": has_neonatal,
        "has_oncology": has_oncology,
        "has_anesthesiologist": has_anesthesiologist,
        "trust_score": score,
        "trust_level": trust_level,
        "trust_flags": flags,
        "citation": str(row.get("capability", "") or "")[:200],
        "extraction_confidence": 0.8
    })

print(f"✅ {len(results)} facilities ready!")

✅ 10000 facilities ready!


In [0]:
import folium
from folium.plugins import HeatMap

m = folium.Map(location=[20.5937, 78.9629], zoom_start=5, tiles="CartoDB positron")
heat_data = []
plotted = 0

for r in results:
    lat = float(r.get("latitude") or 0)
    lon = float(r.get("longitude") or 0)
    if not lat or not lon:
        continue

    trust = int(r.get("trust_score") or 50)
    color = "#2ECC71" if trust >= 75 else "#F39C12" if trust >= 45 else "#E74C3C"

    popup_html = f"""
    <div style='font-family:Arial;width:220px'>
    <b>{r.get('facility_name','')}</b><br>
    📍 {r.get('address_city','')}, {r.get('state','')}<br>
    ⭐ Trust: <b>{trust}/100</b> — {r.get('trust_level','')}<br>
    🚨 Emergency: {'✅' if r.get('has_emergency') else '❌'} |
    🛏️ ICU: {'✅' if r.get('has_icu') else '❌'} |
    🔪 Surgery: {'✅' if r.get('has_surgery') else '❌'}<br>
    📞 {r.get('phone','')}<br>
    <hr style='margin:4px'>
    <i style='font-size:11px'>{str(r.get('citation',''))[:120]}</i>
    </div>"""

    folium.CircleMarker(
        location=[lat, lon], radius=4,
        color=color, fill=True, fill_opacity=0.7,
        popup=folium.Popup(popup_html, max_width=240)
    ).add_to(m)

    if trust < 45:
        heat_data.append([lat, lon])
    plotted += 1

HeatMap(heat_data, name="⚠️ Medical Deserts",
        radius=25, blur=20,
        gradient={0.2:"yellow", 0.5:"orange", 1.0:"red"}).add_to(m)

folium.LayerControl().add_to(m)
print(f"✅ {plotted} facilities plotted!")
print(f"🔴 Desert markers: {len(heat_data)}")
displayHTML(m._repr_html_())

✅ 10000 facilities plotted!
🔴 Desert markers: 121


*** WARNING: Output too large, the following mime types were removed from the output: text/html. ***

In [0]:
import folium
from folium.plugins import HeatMap

m = folium.Map(location=[20.5937, 78.9629], zoom_start=5, tiles="CartoDB positron")
heat_data = []
plotted = 0

for r in results:
    lat = float(r.get("latitude") or 0)
    lon = float(r.get("longitude") or 0)
    if not lat or not lon:
        continue

    trust = int(r.get("trust_score") or 50)
    color = "#2ECC71" if trust >= 75 else "#F39C12" if trust >= 45 else "#E74C3C"

    # Simplified popup to reduce file size
    popup_text = f"{r.get('facility_name','')} | Trust:{trust} | {r.get('trust_level','')}"

    folium.CircleMarker(
        location=[lat, lon], radius=3,
        color=color, fill=True, fill_opacity=0.6,
        tooltip=popup_text
    ).add_to(m)

    if trust < 45:
        heat_data.append([lat, lon])
    plotted += 1

HeatMap(heat_data, name="Medical Deserts",
        radius=20, blur=15,
        gradient={0.2:"yellow", 0.5:"orange", 1.0:"red"}).add_to(m)

folium.LayerControl().add_to(m)

# Save to file
m.save("/tmp/aarogyapath_map.html")
print(f"✅ {plotted} facilities plotted!")
print(f"🔴 Desert markers: {len(heat_data)}")
print(f"📁 Map saved to /tmp/aarogyapath_map.html")

# Copy to DBFS so you can download it
import shutil
shutil.copy("/tmp/aarogyapath_map.html", "/dbfs/FileStore/tables/aarogyapath_map.html")
print(f"\n🌐 Download your map at:")
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
print(f"https://{workspace_url}/files/tables/aarogyapath_map.html")

✅ 10000 facilities plotted!
🔴 Desert markers: 121
📁 Map saved to /tmp/aarogyapath_map.html


---------------------------------------------------------------------------
OSError                                   Traceback (most recent call last)
File <command-7249238166546861>, line 44
     42 # Copy to DBFS so you can download it
     43 import shutil
---> 44 shutil.copy("/tmp/aarogyapath_map.html", "/dbfs/FileStore/tables/aarogyapath_map.html")
     45 print(f"\n🌐 Download your map at:")
     46 workspace_url = spark.conf.get("spark.databricks.workspaceUrl")

File /usr/lib/python3.12/shutil.py:435, in copy(src, dst, follow_symlinks)
    433 if os.path.isdir(dst):
    434     dst = os.path.join(dst, os.path.basename(src))
--> 435 copyfile(src, dst, follow_symlinks=follow_symlinks)
    436 copymode(src, dst, follow_symlinks=follow_symlinks)
    437 return dst

File /usr/lib/python3.12/shutil.py:262, in copyfile(src, dst, follow_symlinks)
    260 with open(src, 'rb') as fsrc:
    261     try:
--> 262         with open(dst, 'wb') as fdst:
    263             # macOS
    264      

In [0]:
m.save("/tmp/aarogyapath_map.html")
print("✅ Map saved!")

# Read and display as HTML
with open("/tmp/aarogyapath_map.html", "r") as f:
    map_html = f.read()

# Display in smaller iframe
displayHTML(f"""
<div style="width:100%;height:500px;overflow:hidden">
{map_html[:500000]}
</div>
""")

✅ Map saved!


<!DOCTYPE html>

In [0]:
import folium
from folium.plugins import HeatMap, MarkerCluster

m = folium.Map(location=[20.5937, 78.9629], zoom_start=5, tiles="CartoDB positron")

# Only plot high trust facilities to keep it light
cluster = MarkerCluster(name="Facilities").add_to(m)
heat_data = []

for r in results:
    lat = float(r.get("latitude") or 0)
    lon = float(r.get("longitude") or 0)
    if not lat or not lon:
        continue

    trust = int(r.get("trust_score") or 50)

    # Only add markers for HIGH trust
    if trust >= 75:
        folium.CircleMarker(
            location=[lat, lon], radius=3,
            color="#2ECC71", fill=True, fill_opacity=0.7,
            tooltip=f"{r.get('facility_name','')} | {r.get('state','')}"
        ).add_to(cluster)

    # Heatmap for deserts
    if trust < 45:
        heat_data.append([lat, lon])

HeatMap(heat_data, radius=20, blur=15,
        gradient={0.2:"yellow",0.5:"orange",1.0:"red"},
        name="Medical Deserts").add_to(m)

folium.LayerControl().add_to(m)
m.save("/tmp/map_light.html")

# Get download link
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

print("✅ Map saved!")
print("\nRun this in terminal or next cell to serve it:")
print("Use the Databricks file browser to download /tmp/map_light.html")
print("\nOR just take screenshot of stats below for demo backup:")
print(f"\n📊 DEMO STATS TO SHOW JUDGES:")
print(f"✅ Total facilities analysed: {len(results)}")
print(f"🚨 Emergency capable: {sum(1 for r in results if r.get('has_emergency'))}")
print(f"🛏️  ICU capable: {sum(1 for r in results if r.get('has_icu'))}")
print(f"🔪 Surgery capable: {sum(1 for r in results if r.get('has_surgery'))}")
print(f"💉 Dialysis capable: {sum(1 for r in results if r.get('has_dialysis'))}")
print(f"✅ HIGH trust facilities: {sum(1 for r in results if r.get('trust_level')=='HIGH')}")
print(f"⚠️  Contradiction flags raised: {sum(len(r.get('trust_flags',[])) for r in results)}")
print(f"🔴 Medical desert zones: {sum(1 for r in results if not r.get('has_emergency') and not r.get('has_icu') and not r.get('has_surgery'))}")

✅ Map saved!

Run this in terminal or next cell to serve it:
Use the Databricks file browser to download /tmp/map_light.html

OR just take screenshot of stats below for demo backup:

📊 DEMO STATS TO SHOW JUDGES:
✅ Total facilities analysed: 10000
🚨 Emergency capable: 1208
🛏️  ICU capable: 330
🔪 Surgery capable: 2342
💉 Dialysis capable: 227
✅ HIGH trust facilities: 7751
⚠️  Contradiction flags raised: 9359
🔴 Medical desert zones: 6844


In [0]:
import json

# Save facilities to file for Flask
with open("/tmp/facilities.json", "w") as f:
    json.dump(results, f)
print(f"✅ {len(results)} facilities saved to /tmp/facilities.json")

# Write Flask app
app_code = '''
from flask import Flask, request, jsonify
from flask_cors import CORS
import json, math, os, re, requests

app = Flask(__name__)
CORS(app)

with open("/tmp/facilities.json") as f:
    FACILITIES = json.load(f)

print(f"Loaded {len(FACILITIES)} facilities")

DATABRICKS_TOKEN = os.environ.get("DATABRICKS_TOKEN","")
WORKSPACE_URL = os.environ.get("WORKSPACE_URL","")

def call_llm(prompt):
    r = requests.post(
        f"https://{WORKSPACE_URL}/serving-endpoints/databricks-llama-4-maverick/invocations",
        headers={"Authorization": f"Bearer {DATABRICKS_TOKEN}"},
        json={"messages":[{"role":"user","content":prompt}],"max_tokens":600}
    )
    return r.json()["choices"][0]["message"]["content"]

def haversine(lat1,lon1,lat2,lon2):
    R=6371
    p1,p2=math.radians(lat1),math.radians(lat2)
    dp=math.radians(lat2-lat1)
    dl=math.radians(lon2-lon1)
    a=math.sin(dp/2)**2+math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return R*2*math.atan2(math.sqrt(a),math.sqrt(1-a))

@app.route("/triage", methods=["POST"])
def triage():
    body = request.json
    symptoms = body.get("symptoms","")
    location = body.get("location","India")
    prompt = f"""You are AarogyaPath emergency triage for rural India.
Patient location: {location}
Symptoms: {symptoms}
Return ONLY valid JSON no markdown:
{{
  "severity": "EMERGENCY or URGENT or MILD",
  "severity_reason": "one sentence why",
  "action": "what to do RIGHT NOW",
  "facility_type_needed": "ICU or Emergency or GeneralSurgery or Dialysis or Clinic or Homecare",
  "call_ambulance": true or false,
  "home_care_steps": ["step1","step2","step3"],
  "red_flags": ["flag1","flag2"],
  "chain_of_thought": "your reasoning"
}}"""
    raw = call_llm(prompt).strip()
    match = re.search(r"\\{.*\\}", raw, re.DOTALL)
    if match: raw = match.group()
    return jsonify(json.loads(raw))

@app.route("/facilities", methods=["POST"])
def find_facilities():
    body = request.json
    patient_lat = float(body.get("lat", 20.5))
    patient_lon = float(body.get("lon", 78.9))
    needed_type = body.get("facility_type","Emergency")
    scored = []
    for f in FACILITIES:
        try:
            lat = float(f.get("latitude") or 0)
            lon = float(f.get("longitude") or 0)
            if not lat or not lon: continue
            dist = haversine(patient_lat, patient_lon, lat, lon)
            trust = int(f.get("trust_score") or 50)
            reachability = max(0, 100 - dist*0.5)
            cap_map = {
                "ICU":["has_icu","has_emergency"],
                "Emergency":["has_emergency"],
                "GeneralSurgery":["has_surgery","has_anesthesiologist"],
                "Dialysis":["has_dialysis"],
            }
            required = cap_map.get(needed_type,[])
            cap_score = (sum(1 for k in required if f.get(k)==True)/len(required)*100) if required else 70
            last_mile = (trust*0.45)+(reachability*0.30)+(cap_score*0.25)
            scored.append({
                "facility_name": f.get("facility_name",""),
                "city": f.get("address_city",""),
                "state": f.get("state",""),
                "phone": f.get("phone",""),
                "distance_km": round(dist,1),
                "trust_score": trust,
                "trust_level": f.get("trust_level",""),
                "trust_flags": f.get("trust_flags",[]),
                "last_mile_score": round(last_mile,1),
                "has_emergency": f.get("has_emergency"),
                "has_icu": f.get("has_icu"),
                "has_surgery": f.get("has_surgery"),
                "citation": f.get("citation",""),
                "verdict": "RECOMMENDED" if last_mile>=70 else "USE IF NEEDED" if last_mile>=45 else "AVOID",
                "latitude": lat,
                "longitude": lon
            })
        except: continue
    scored.sort(key=lambda x: x["last_mile_score"], reverse=True)
    return jsonify({"facilities": scored[:5]})

@app.route("/deserts", methods=["GET"])
def deserts():
    from collections import defaultdict
    pins = defaultdict(lambda: {"missing":[],"state":"","lat":0,"lon":0})
    for f in FACILITIES:
        if int(f.get("trust_score") or 0) < 45: continue
        pin = str(f.get("pincode",""))
        missing = []
        if not f.get("has_icu"): missing.append("ICU")
        if not f.get("has_emergency"): missing.append("Emergency")
        if not f.get("has_surgery"): missing.append("Surgery")
        if not f.get("has_dialysis"): missing.append("Dialysis")
        if len(missing) >= 3:
            pins[pin]["missing"] = missing
            pins[pin]["state"] = f.get("state","")
            pins[pin]["lat"] = float(f.get("latitude") or 0)
            pins[pin]["lon"] = float(f.get("longitude") or 0)
    deserts_list = [
        {"pincode":k,"state":v["state"],
         "missing_capabilities":v["missing"],
         "severity":"CRITICAL" if len(v["missing"])>=4 else "HIGH",
         "lat":v["lat"],"lon":v["lon"]}
        for k,v in pins.items()
    ]
    deserts_list.sort(key=lambda x: len(x["missing_capabilities"]), reverse=True)
    return jsonify({
        "total_desert_pins": len(deserts_list),
        "critical_count": sum(1 for d in deserts_list if d["severity"]=="CRITICAL"),
        "deserts": deserts_list[:100]
    })

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status":"ok","facilities":len(FACILITIES)})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=False)
'''

with open("/tmp/app.py", "w") as f:
    f.write(app_code)
print("✅ app.py written!")

✅ 10000 facilities saved to /tmp/facilities.json
✅ app.py written!


In [0]:
import os

app_code = """from flask import Flask, request, jsonify
from flask_cors import CORS
import json, math, os, re, requests

app = Flask(__name__)
CORS(app)

with open("/tmp/facilities.json") as f:
    FACILITIES = json.load(f)

print(f"Loaded {len(FACILITIES)} facilities")

DATABRICKS_TOKEN = os.environ.get("DATABRICKS_TOKEN","")
WORKSPACE_URL = os.environ.get("WORKSPACE_URL","")

def call_llm(prompt):
    r = requests.post(
        f"https://{WORKSPACE_URL}/serving-endpoints/databricks-llama-4-maverick/invocations",
        headers={"Authorization": f"Bearer {DATABRICKS_TOKEN}"},
        json={"messages":[{"role":"user","content":prompt}],"max_tokens":600}
    )
    return r.json()["choices"][0]["message"]["content"]

def haversine(lat1,lon1,lat2,lon2):
    R=6371
    p1,p2=math.radians(lat1),math.radians(lat2)
    dp=math.radians(lat2-lat1)
    dl=math.radians(lon2-lon1)
    a=math.sin(dp/2)**2+math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return R*2*math.atan2(math.sqrt(a),math.sqrt(1-a))

@app.route("/triage", methods=["POST"])
def triage():
    body = request.json
    symptoms = body.get("symptoms","")
    location = body.get("location","India")
    prompt = f"You are AarogyaPath emergency triage for rural India. Patient location: {location}. Symptoms: {symptoms}. Return ONLY valid JSON: {{\\\"severity\\\": \\\"EMERGENCY or URGENT or MILD\\\", \\\"severity_reason\\\": \\\"one sentence\\\", \\\"action\\\": \\\"what to do RIGHT NOW\\\", \\\"facility_type_needed\\\": \\\"ICU or Emergency or GeneralSurgery or Dialysis or Clinic or Homecare\\\", \\\"call_ambulance\\\": true or false, \\\"home_care_steps\\\": [\\\"step1\\\",\\\"step2\\\"], \\\"red_flags\\\": [\\\"flag1\\\"], \\\"chain_of_thought\\\": \\\"reasoning\\\"}}"
    raw = call_llm(prompt).strip()
    match = re.search(r"\\{.*\\}", raw, re.DOTALL)
    if match:
        raw = match.group()
    return jsonify(json.loads(raw))

@app.route("/facilities", methods=["POST"])
def find_facilities():
    body = request.json
    patient_lat = float(body.get("lat", 20.5))
    patient_lon = float(body.get("lon", 78.9))
    needed_type = body.get("facility_type","Emergency")
    scored = []
    for f in FACILITIES:
        try:
            lat = float(f.get("latitude") or 0)
            lon = float(f.get("longitude") or 0)
            if not lat or not lon:
                continue
            dist = haversine(patient_lat, patient_lon, lat, lon)
            trust = int(f.get("trust_score") or 50)
            reachability = max(0, 100 - dist*0.5)
            cap_map = {"ICU":["has_icu","has_emergency"],"Emergency":["has_emergency"],"GeneralSurgery":["has_surgery","has_anesthesiologist"],"Dialysis":["has_dialysis"]}
            required = cap_map.get(needed_type,[])
            cap_score = (sum(1 for k in required if f.get(k)==True)/len(required)*100) if required else 70
            last_mile = (trust*0.45)+(reachability*0.30)+(cap_score*0.25)
            scored.append({
                "facility_name": f.get("facility_name",""),
                "city": f.get("address_city",""),
                "state": f.get("state",""),
                "phone": f.get("phone",""),
                "distance_km": round(dist,1),
                "trust_score": trust,
                "trust_level": f.get("trust_level",""),
                "trust_flags": f.get("trust_flags",[]),
                "last_mile_score": round(last_mile,1),
                "has_emergency": f.get("has_emergency"),
                "has_icu": f.get("has_icu"),
                "has_surgery": f.get("has_surgery"),
                "citation": f.get("citation",""),
                "verdict": "RECOMMENDED" if last_mile>=70 else "USE IF NEEDED" if last_mile>=45 else "AVOID",
                "latitude": lat,
                "longitude": lon
            })
        except:
            continue
    scored.sort(key=lambda x: x["last_mile_score"], reverse=True)
    return jsonify({"facilities": scored[:5]})

@app.route("/deserts", methods=["GET"])
def deserts():
    from collections import defaultdict
    pins = defaultdict(lambda: {"missing":[],"state":"","lat":0,"lon":0})
    for f in FACILITIES:
        if int(f.get("trust_score") or 0) < 45:
            continue
        pin = str(f.get("pincode",""))
        missing = []
        if not f.get("has_icu"): missing.append("ICU")
        if not f.get("has_emergency"): missing.append("Emergency")
        if not f.get("has_surgery"): missing.append("Surgery")
        if not f.get("has_dialysis"): missing.append("Dialysis")
        if len(missing) >= 3:
            pins[pin]["missing"] = missing
            pins[pin]["state"] = f.get("state","")
            pins[pin]["lat"] = float(f.get("latitude") or 0)
            pins[pin]["lon"] = float(f.get("longitude") or 0)
    deserts_list = [{"pincode":k,"state":v["state"],"missing_capabilities":v["missing"],"severity":"CRITICAL" if len(v["missing"])>=4 else "HIGH","lat":v["lat"],"lon":v["lon"]} for k,v in pins.items()]
    deserts_list.sort(key=lambda x: len(x["missing_capabilities"]), reverse=True)
    return jsonify({"total_desert_pins":len(deserts_list),"critical_count":sum(1 for d in deserts_list if d["severity"]=="CRITICAL"),"deserts":deserts_list[:100]})

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status":"ok","facilities":len(FACILITIES)})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=False)
"""

with open("/tmp/app.py", "w") as f:
    f.write(app_code)
print("✅ app.py written cleanly!")

# Verify it
import py_compile
try:
    py_compile.compile("/tmp/app.py", doraise=True)
    print("✅ Syntax check passed!")
except py_compile.PyCompileError as e:
    print(f"❌ Syntax error: {e}")

✅ app.py written cleanly!
✅ Syntax check passed!


In [0]:
import requests, json, os, math, time, threading, subprocess
from io import BytesIO
import pandas as pd

# ── 1. Reload dataset ──
SHEET_ID = "1ZDuDmoQlyxZIEahDBlrMjf2wiWG7xU81"
df = pd.read_excel(BytesIO(requests.get(f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=xlsx").content))
print(f"✅ {len(df)} rows loaded")

# ── 2. Rebuild results instantly ──
results = []
for _, row in df.iterrows():
    t = " ".join([str(row.get(c,"") or "") for c in ["capability","specialties","description","procedure","equipment"]]).lower()
    has_emergency = any(w in t for w in ["emergency","24x7","24/7","always open","round the clock","casualty"])
    has_icu       = any(w in t for w in ["icu","intensive care","critical care","ventilator"])
    has_surgery   = any(w in t for w in ["surgery","surgical","operation","theatre","appendectomy","laparoscop"])
    has_dialysis  = any(w in t for w in ["dialysis","renal","kidney","nephrology"])
    has_neonatal  = any(w in t for w in ["neonatal","nicu","newborn","maternity","obstetric"])
    has_oncology  = any(w in t for w in ["oncology","cancer","chemotherapy","radiation"])
    has_anesthesiologist = any(w in t for w in ["anesthes","anaesth"])
    score, flags = 100, []
    if has_surgery and not has_anesthesiologist:
        score -= 40; flags.append("CRITICAL: Surgery claimed but no Anesthesiologist")
    if has_icu and "bed" not in t:
        score -= 20; flags.append("WARNING: ICU claimed but no beds mentioned")
    if not has_emergency and not has_icu and not has_surgery:
        score -= 15; flags.append("INFO: No high-acuity capabilities found")
    trust_level = "HIGH" if score>=75 else "MEDIUM" if score>=45 else "LOW"
    results.append({
        "facility_name": str(row.get("name","")),
        "address_city":  str(row.get("address_city","")),
        "state":         str(row.get("address_stateOrRegion","")),
        "pincode":       str(row.get("address_zipOrPostcode","")),
        "latitude":      float(row.get("latitude",0) or 0),
        "longitude":     float(row.get("longitude",0) or 0),
        "phone":         str(row.get("officialPhone","") or ""),
        "facility_type": str(row.get("facilityTypeId","") or ""),
        "has_emergency": has_emergency, "has_icu": has_icu,
        "has_surgery": has_surgery, "has_dialysis": has_dialysis,
        "has_neonatal": has_neonatal, "has_oncology": has_oncology,
        "has_anesthesiologist": has_anesthesiologist,
        "trust_score": score, "trust_level": trust_level,
        "trust_flags": flags,
        "citation": str(row.get("capability","") or "")[:200],
        "extraction_confidence": 0.8
    })

print(f"✅ {len(results)} facilities extracted!")
print(f"🚨 Emergency: {sum(1 for r in results if r.get('has_emergency'))}")
print(f"🛏️  ICU:       {sum(1 for r in results if r.get('has_icu'))}")
print(f"🔪 Surgery:   {sum(1 for r in results if r.get('has_surgery'))}")
print(f"✅ HIGH trust: {sum(1 for r in results if r.get('trust_level')=='HIGH')}")

# ── 3. Save facilities.json ──
with open("/tmp/facilities.json","w") as f:
    json.dump(results, f)
print(f"✅ facilities.json saved!")

# ── 4. Write app.py ──
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
WORKSPACE_URL = spark.conf.get("spark.databricks.workspaceUrl")
os.environ["DATABRICKS_TOKEN"] = DATABRICKS_TOKEN
os.environ["WORKSPACE_URL"] = WORKSPACE_URL

with open("/tmp/app.py","w") as f:
    f.write("""
import os,json,math,re,requests
from flask import Flask,request,jsonify
from flask_cors import CORS
app=Flask(__name__)
CORS(app)
with open("/tmp/facilities.json") as f: FACILITIES=json.load(f)
TOKEN=os.environ.get("DATABRICKS_TOKEN","")
WS=os.environ.get("WORKSPACE_URL","")
def llm(p):
    r=requests.post(f"https://{WS}/serving-endpoints/databricks-llama-4-maverick/invocations",headers={"Authorization":f"Bearer {TOKEN}"},json={"messages":[{"role":"user","content":p}],"max_tokens":600})
    return r.json()["choices"][0]["message"]["content"]
def dist(a,b,c,d):
    R=6371;p1,p2=math.radians(a),math.radians(c)
    x=math.sin((c-a)/2*math.pi/180)**2+math.cos(p1)*math.cos(p2)*math.sin((d-b)/2*math.pi/180)**2
    return R*2*math.atan2(math.sqrt(x),math.sqrt(1-x))
@app.route("/health")
def health(): return jsonify({"status":"ok","facilities":len(FACILITIES)})
@app.route("/triage",methods=["POST"])
def triage():
    b=request.json
    raw=llm(f'Triage for rural India. Location:{b.get("location","India")}. Symptoms:{b.get("symptoms","")}. Return ONLY JSON:{{"severity":"EMERGENCY or URGENT or MILD","severity_reason":"why","action":"what to do now","facility_type_needed":"ICU or Emergency or GeneralSurgery or Dialysis or Clinic or Homecare","call_ambulance":true,"home_care_steps":["step1"],"red_flags":["flag1"],"chain_of_thought":"reasoning"}}').strip()
    m=re.search(r'\\{.*\\}',raw,re.DOTALL)
    return jsonify(json.loads(m.group() if m else raw))
@app.route("/facilities",methods=["POST"])
def facilities():
    b=request.json
    plat,plon=float(b.get("lat",20.5)),float(b.get("lon",78.9))
    nt=b.get("facility_type","Emergency")
    cm={"ICU":["has_icu","has_emergency"],"Emergency":["has_emergency"],"GeneralSurgery":["has_surgery","has_anesthesiologist"],"Dialysis":["has_dialysis"]}
    req=cm.get(nt,[])
    scored=[]
    for f in FACILITIES:
        try:
            la,lo=float(f.get("latitude") or 0),float(f.get("longitude") or 0)
            if not la or not lo: continue
            d=dist(plat,plon,la,lo);t=int(f.get("trust_score") or 50)
            r=max(0,100-d*0.5);c=(sum(1 for k in req if f.get(k)==True)/len(req)*100) if req else 70
            lm=t*0.45+r*0.30+c*0.25
            scored.append({"facility_name":f.get("facility_name",""),"city":f.get("address_city",""),"state":f.get("state",""),"phone":f.get("phone",""),"distance_km":round(d,1),"trust_score":t,"trust_level":f.get("trust_level",""),"trust_flags":f.get("trust_flags",[]),"last_mile_score":round(lm,1),"has_icu":f.get("has_icu"),"has_emergency":f.get("has_emergency"),"has_surgery":f.get("has_surgery"),"citation":f.get("citation",""),"verdict":"RECOMMENDED" if lm>=70 else "USE IF NEEDED" if lm>=45 else "AVOID","latitude":la,"longitude":lo})
        except: continue
    scored.sort(key=lambda x:x["last_mile_score"],reverse=True)
    return jsonify({"facilities":scored[:5]})
@app.route("/survivability",methods=["POST"])
def survivability():
    b=request.json
    plat,plon=float(b.get("lat",20.5)),float(b.get("lon",78.9))
    nt=b.get("facility_type","Emergency")
    win={"ICU":90,"Emergency":60,"GeneralSurgery":360,"Dialysis":720,"Neonatal":30}.get(nt,60)
    key={"ICU":"has_icu","Emergency":"has_emergency","GeneralSurgery":"has_surgery","Dialysis":"has_dialysis"}.get(nt,"has_emergency")
    q=[f for f in FACILITIES if f.get(key) and int(f.get("trust_score") or 0)>=45 and float(f.get("latitude") or 0)!=0]
    if not q: return jsonify({"survivability":"UNKNOWN"})
    for f in q: f["_d"]=dist(plat,plon,float(f.get("latitude") or 0),float(f.get("longitude") or 0))
    q.sort(key=lambda x:x["_d"]); n=q[0]
    travel=n["_d"]/35*60; margin=win-travel
    sv="GOOD" if margin>=30 else "TIGHT" if margin>=0 else "CRITICAL" if margin>=-60 else "FATAL_RISK"
    return jsonify({"survivability":sv,"color":"🟢" if sv=="GOOD" else "🟡" if sv=="TIGHT" else "🟠" if sv=="CRITICAL" else "🔴","survival_window_minutes":win,"nearest_name":n.get("facility_name",""),"nearest_city":n.get("address_city",""),"distance_km":round(n["_d"],1),"travel_minutes":round(travel,1),"margin_minutes":round(margin,1),"facilities_in_window":len([f for f in q if f["_d"]/35*60<=win]),"verdict":f"Within window — {int(margin)} min to spare" if margin>=0 else f"Window exceeded by {int(abs(margin))} min"})
@app.route("/deserts")
def deserts():
    from collections import defaultdict
    pins=defaultdict(lambda:{"missing":[],"state":"","lat":0,"lon":0})
    for f in FACILITIES:
        if int(f.get("trust_score") or 0)<45: continue
        pin=str(f.get("pincode",""));m=[]
        if not f.get("has_icu"): m.append("ICU")
        if not f.get("has_emergency"): m.append("Emergency")
        if not f.get("has_surgery"): m.append("Surgery")
        if not f.get("has_dialysis"): m.append("Dialysis")
        if len(m)>=3: pins[pin]={"missing":m,"state":f.get("state",""),"lat":float(f.get("latitude") or 0),"lon":float(f.get("longitude") or 0)}
    dl=[{"pincode":k,"state":v["state"],"missing_capabilities":v["missing"],"severity":"CRITICAL" if len(v["missing"])>=4 else "HIGH","lat":v["lat"],"lon":v["lon"]} for k,v in pins.items()]
    dl.sort(key=lambda x:len(x["missing_capabilities"]),reverse=True)
    return jsonify({"total_desert_pins":len(dl),"critical_count":sum(1 for d in dl if d["severity"]=="CRITICAL"),"deserts":dl[:100]})
app.run(host="0.0.0.0",port=5000,debug=False)
""")
print("✅ app.py written!")

# ── 5. Start Flask ──
proc = subprocess.Popen(["python","/tmp/app.py"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)

try:
    import urllib.request
    resp = urllib.request.urlopen("http://localhost:5000/health", timeout=3)
    print(f"\n✅ FLASK IS RUNNING! {resp.read().decode()}")
    print("\nNow run Cell 2 with your ngrok token!")
except Exception as e:
    out,err = proc.stdout.read(500), proc.stderr.read(500)
    print(f"❌ Error: {e}")
    print(f"stderr: {err.decode()}")

✅ 10000 rows loaded
✅ 10000 facilities extracted!
🚨 Emergency: 1216
🛏️  ICU:       347
🔪 Surgery:   2389
✅ HIGH trust: 7720
✅ facilities.json saved!
✅ app.py written!

✅ FLASK IS RUNNING! {"facilities":10000,"status":"ok"}


Now run Cell 2 with your ngrok token!


In [0]:
import subprocess, time, os, threading

DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
WORKSPACE_URL = spark.conf.get("spark.databricks.workspaceUrl")
os.environ["DATABRICKS_TOKEN"] = DATABRICKS_TOKEN
os.environ["WORKSPACE_URL"] = WORKSPACE_URL

# Kill old Flask
os.system("pkill -f 'python /tmp/app.py' 2>/dev/null")
time.sleep(2)

# Start Flask
proc = subprocess.Popen(
    ["python", "/tmp/app.py"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    env=os.environ.copy()
)
time.sleep(5)

# Verify Flask
import urllib.request
try:
    r = urllib.request.urlopen("http://localhost:5000/health", timeout=3)
    print(f"✅ Flask running: {r.read().decode()}")
except Exception as e:
    print(f"❌ Flask failed: {e}")
    raise

# Use localhost.run tunnel via SSH (no install needed)
tunnel_proc = subprocess.Popen(
    ["ssh", "-o", "StrictHostKeyChecking=no", "-R", "80:localhost:5000", "nokey@localhost.run"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True
)

# Read URL from output
print("Getting tunnel URL...")
for i in range(15):
    line = tunnel_proc.stdout.readline()
    print(line.strip())
    if "localhost.run" in line and "https://" in line:
        import re
        urls = re.findall(r'https://[a-zA-Z0-9\-]+\.localhost\.run', line)
        if urls:
            PUBLIC_URL = urls[0]
            print(f"\n🌐 YOUR LIVE URL: {PUBLIC_URL}")
            print(f"Test: {PUBLIC_URL}/health")
            print(f"\n✅ Paste this into Lovable!")
            break
    time.sleep(1)

✅ Flask running: {"facilities":10000,"status":"ok"}

Getting tunnel URL...
Pseudo-terminal will not be allocated because stdin is not a terminal.

Welcome to localhost.run!

Follow your favourite reverse tunnel at [https://twitter.com/localhost_run].

To set up and manage custom domains go to https://admin.localhost.run/

🌐 YOUR LIVE URL: https://admin.localhost.run
Test: https://admin.localhost.run/health

✅ Paste this into Lovable!


In [0]:
# Run this to get the real tunnel URL
print("Waiting for real tunnel URL...")
for i in range(20):
    line = tunnel_proc.stdout.readline()
    if line.strip():
        print(repr(line.strip()))
    time.sleep(1)

Waiting for real tunnel URL...
'More details on custom domains (and how to enable subdomains of your custom'
'domain) at https://localhost.run/docs/custom-domains'
'If you get a permission denied error check the faq for how to connect with a key or'
'create a free tunnel without a key at [http://localhost:3000/docs/faq#generating-an-ssh-key].'
'To explore using localhost.run visit the documentation site:'
'https://localhost.run/docs/'
'==============================================================================='
'** your connection id is a7846a6c-ee4e-4b05-90ce-64730cab170d, please mention it if you send me a message about an issue. **'
'authenticated as anonymous user'
'e05f319fdf1fd7.lhr.life tunneled with tls termination, https://e05f319fdf1fd7.lhr.life'
'create an account and add your key for a longer lasting domain name. see https://localhost.run/docs/forever-free/ for more information.'
'Open your tunnel address on your mobile with this QR:'
'\x1b  \x1b\x1b  \x1b\x1b  \x1b\x1b